# Batch Ingestion and Data Quality

This notebook covers **Task 1** (batch ingestion pipeline) and **Task 2** (data quality checks).

**Environment:** Databricks Free Edition, serverless, PySpark.
**Input:** raw `orders.csv` and `customers.csv` from a Unity Catalog volume.
**Output:** cleaned Parquet, partitioned by order year and month, plus a data quality report.

Approach: read the raw files as text, check what is actually wrong, clean each issue, run quality checks, then write the cleaned output.

In [0]:
# Task 1: batch ingestion pipeline
# read raw orders and customers, clean them, run quality checks,
# write cleaned tables partitioned by order year and month.

from pyspark.sql import functions as F, Window
from pyspark.sql.types import DecimalType

RAW = "/Volumes/workspace/catenaretail/raw"
ORDERS_RAW_PATH = f"{RAW}/orders.csv"
CUSTOMERS_RAW_PATH = f"{RAW}/customers.csv"

# read everything as strings so we control the type casting ourselves
customers_raw = (spark.read.option("header", True).option("inferSchema", False)
                 .csv(CUSTOMERS_RAW_PATH))
orders_raw = (spark.read.option("header", True).option("inferSchema", False)
              .csv(ORDERS_RAW_PATH))

raw_cust = customers_raw.count()
raw_ord = orders_raw.count()
print(f"[RAW] customers = {raw_cust} rows")
print(f"[RAW] orders = {raw_ord} rows")

display(orders_raw.limit(5))

## Profiling: what's actually wrong in the raw data

In [0]:
# check the data for problems first, then clean based on what we find

print("customers issues")
print(" duplicate rows:", customers_raw.count() - customers_raw.dropDuplicates().count())
print(" duplicate customer_id:", customers_raw.groupBy("customer_id").count().filter("count > 1").count())
print(" null region:", customers_raw.filter("region is null").count())

print("\norders issues")
print(" duplicate rows:", orders_raw.count() - orders_raw.dropDuplicates().count())
print(" duplicate order_id:", orders_raw.groupBy("order_id").count().filter("count > 1").count())
print(" null amount:", orders_raw.filter("amount is null").count())

bad_amount = orders_raw.filter(~F.col("amount").rlike(r"^[0-9]+\.?[0-9]*$"))
print(" bad amount values:", [r["amount"] for r in bad_amount.select("amount").distinct().collect()])

bad_date = orders_raw.filter(~F.col("order_date").rlike(r"^\d{4}-\d{2}-\d{2}$"))
print(" bad date values:", [r["order_date"] for r in bad_date.select("order_date").distinct().collect()])

print(" bad quantity values:", [r["quantity"] for r in orders_raw.filter("cast(quantity as int) <= 0").select("quantity").distinct().collect()])

## Clean customers
Fix region, parse dates safely, drop rows with no id, remove duplicate ids.

In [0]:
# clean customers: fix region, parse date, drop rows with no id, remove duplicate ids

valid_regions = ["North", "South", "East", "West", "Central"]

customers_clean = (customers_raw
    .withColumn("customer_id", F.trim("customer_id"))
    .withColumn("customer_name", F.trim("customer_name"))
    .withColumn("region", F.trim("region"))
    # keep the customer but mark unknown regions instead of dropping them
    .withColumn("region", F.when(F.col("region").isin(valid_regions), F.col("region"))
                           .otherwise(F.lit("Unknown")))
    # try_to_date tolerates bad dates like 2023-13-40 and returns null instead of failing
    .withColumn("signup_date", F.expr("try_to_date(signup_date, 'yyyy-MM-dd')"))
    .filter(F.col("customer_id").isNotNull() & (F.col("customer_id") != "")))

# drop duplicate customer_id, keep one row per id deterministically
w = Window.partitionBy("customer_id").orderBy(F.col("signup_date").asc_nulls_last())
customers_clean = (customers_clean
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .drop("rn"))

clean_cust = customers_clean.count()
print(f"[CLEANED] customers = {clean_cust} rows (dropped {raw_cust - clean_cust})")
display(customers_clean.filter("region = 'Unknown'"))

## Clean orders
Fix currency and dates


In [0]:
# clean orders: fix currency strings, parse mixed date formats, cast numbers,
# drop rows we cannot trust, remove duplicates, flag orphaned customers

orders_clean = (orders_raw
    .withColumn("order_id", F.trim("order_id"))
    .withColumn("customer_id", F.trim("customer_id"))
    .withColumn("product_id", F.trim("product_id"))
    .withColumn("product_name", F.trim("product_name"))
    # strip anything that is not a digit, dot or minus, then cast to decimal
    .withColumn("amount", F.regexp_replace("amount", r"[^0-9.\-]", "").cast(DecimalType(12, 2)))
    .withColumn("quantity", F.col("quantity").cast("int"))
    # data has two date formats: yyyy-MM-dd and MM/dd/yyyy. try both.
    .withColumn("order_date", F.coalesce(
        F.expr("try_to_date(order_date, 'yyyy-MM-dd')"),
        F.expr("try_to_date(order_date, 'MM/dd/yyyy')"))))

# quarantine rows that cannot be used for analytics
orders_clean = orders_clean.filter(
    (F.col("quantity") > 0) &
    F.col("amount").isNotNull() &
    F.col("order_date").isNotNull() &
    F.col("order_id").isNotNull())

# remove exact duplicate rows first
orders_clean = orders_clean.dropDuplicates()

# for same order_id with conflicting values, keep the highest amount (deterministic)
w = Window.partitionBy("order_id").orderBy(F.col("amount").desc())
orders_clean = (orders_clean
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .drop("rn"))

# flag orders whose customer_id is not in the customers table
valid_ids = customers_clean.select("customer_id").withColumn("valid", F.lit(True))
orders_clean = (orders_clean.join(valid_ids, "customer_id", "left")
    .withColumn("is_orphan_customer", F.col("valid").isNull())
    .drop("valid"))

# partition columns from the order date
orders_clean = (orders_clean
    .withColumn("order_year", F.year("order_date"))
    .withColumn("order_month", F.month("order_date")))

clean_ord = orders_clean.count()
print(f"[CLEANED] orders = {clean_ord} rows (removed {raw_ord - clean_ord})")
print(" orphaned customer refs kept and flagged:", orders_clean.filter("is_orphan_customer = true").count())
display(orders_clean.limit(5))

## Task 2: data quality checks
Schema, nulls, referential integrity, duplicates. Each returns pass or fail.

In [0]:
# task 2: data quality checks on the cleaned data
# each check returns pass or fail with a short detail, collected into a report

def result(name, passed, detail=""):
    return {"check": name, "status": "PASS" if passed else "FAIL", "detail": detail}

checks = []

# 1. schema conformity: expected columns present in raw input
expected_orders = ["order_id","customer_id","product_id","product_name","quantity","amount","order_date"]
expected_customers = ["customer_id","customer_name","region","signup_date"]
missing_o = [c for c in expected_orders if c not in orders_raw.columns]
missing_c = [c for c in expected_customers if c not in customers_raw.columns]
checks.append(result("schema_orders", len(missing_o)==0, "all columns present" if not missing_o else f"missing {missing_o}"))
checks.append(result("schema_customers", len(missing_c)==0, "all columns present" if not missing_c else f"missing {missing_c}"))

# 2. null checks on key business fields (after cleaning)
key_o = ["order_id","customer_id","amount","order_date"]
nulls_o = {c: orders_clean.filter(F.col(c).isNull()).count() for c in key_o}
bad_o = {c:n for c,n in nulls_o.items() if n>0}
checks.append(result("nulls_orders_keys", len(bad_o)==0, "no nulls in key fields" if not bad_o else f"nulls: {bad_o}"))

null_cust_id = customers_clean.filter("customer_id is null").count()
checks.append(result("nulls_customer_id", null_cust_id==0, "no null customer_id" if null_cust_id==0 else f"{null_cust_id} nulls"))

# 3. referential integrity: orders pointing to a missing customer
orphans = orders_clean.filter("is_orphan_customer = true").count()
checks.append(result("referential_integrity", orphans==0,
                      "all orders map to a customer" if orphans==0 else f"{orphans} orphaned order(s)"))

# 4. duplicate detection on cleaned keys
dup_o = orders_clean.groupBy("order_id").count().filter("count > 1").count()
dup_c = customers_clean.groupBy("customer_id").count().filter("count > 1").count()
checks.append(result("dupes_order_id", dup_o==0, "no duplicate order_id" if dup_o==0 else f"{dup_o} dupes"))
checks.append(result("dupes_customer_id", dup_c==0, "no duplicate customer_id" if dup_c==0 else f"{dup_c} dupes"))

for c in checks:
    print(f"[{c['status']}] {c['check']}: {c['detail']}")

## Create the output volume for cleaned data and the report.

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.catenaretail.output")
print("output volume ready")

## Write the data quality report

In [0]:
from datetime import datetime, timezone

passed = sum(1 for c in checks if c["status"] == "PASS")
total = len(checks)

lines = []
lines.append("CatenaRetail Data Quality Report")
lines.append(f"generated: {datetime.now(timezone.utc).isoformat()}")
lines.append(f"summary: {passed}/{total} checks passed")
lines.append("")
for c in checks:
    lines.append(f"[{c['status']}] {c['check']}")
    lines.append(f"    {c['detail']}")
lines.append("")
lines.append("note: referential_integrity fails by design. one order references a")
lines.append("customer id not present in the customers file (C9999). the record is")
lines.append("kept and flagged rather than dropped, so revenue is not silently lost.")

report = "\n".join(lines)
print(report)

report_path = "/Volumes/workspace/catenaretail/output/data_quality_report.txt"
with open(report_path, "w") as f:
    f.write(report)
print("\nwritten to:", report_path)

## Write cleaned output

In [0]:
# task 1 final step: write cleaned data as parquet, partitioned by order year and month
# then read it back and log the written counts so the run is auditable

OUT = "/Volumes/workspace/catenaretail/output"
orders_out = f"{OUT}/orders_cleaned"
customers_out = f"{OUT}/customers_cleaned"

(orders_clean.write.mode("overwrite")
    .partitionBy("order_year", "order_month")
    .parquet(orders_out))

customers_clean.write.mode("overwrite").parquet(customers_out)

written_ord = spark.read.parquet(orders_out).count()
written_cust = spark.read.parquet(customers_out).count()

print("audit summary  raw -> cleaned -> written")
print(f" customers: {raw_cust} -> {clean_cust} -> {written_cust}")
print(f" orders:    {raw_ord} -> {clean_ord} -> {written_ord}")
print(f"\norders written to: {orders_out} (partitioned by order_year/order_month)")

## Confirm partitioning

In [0]:
# confirm the parquet output really is partitioned by year then month
import os
for year_dir in sorted(os.listdir(orders_out)):
    if year_dir.startswith("order_year="):
        print(year_dir)
        for month_dir in sorted(os.listdir(f"{orders_out}/{year_dir}")):
            if month_dir.startswith("order_month="):
                print("  ", month_dir)